# Build Dense Embedding Indexes (Colab)

Genera indici dense HNSW per PoliMillionaire usando `sentence-transformers/multi-qa-MiniLM-L6-cos-v1`.

Struttura Drive attesa:

```text
/content/drive/MyDrive/nlp26/
  chunks/simplewiki_160w.jsonl
  kelm/kelm_subset_500k.jsonl
  indexes/
  logs/
  src/
```

Abilita GPU in Colab: `Runtime > Change runtime type > T4 GPU`.

In [ ]:
!pip -q install sentence-transformers hnswlib joblib numpy

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import shutil

PROJECT_ROOT = Path('/content/drive/MyDrive/nlp26')
INDEX_DIR = PROJECT_ROOT / 'indexes'
INDEX_DIR.mkdir(parents=True, exist_ok=True)

SIMPLEWIKI_CORPUS_PATH = PROJECT_ROOT / 'chunks' / 'simplewiki_160w.jsonl'
KELM_CORPUS_PATH = PROJECT_ROOT / 'kelm' / 'kelm_subset_500k.jsonl'

SIMPLEWIKI_DENSE_INDEX_PATH = INDEX_DIR / 'simplewiki_160w_dense_hnsw.index'
SIMPLEWIKI_DENSE_META_PATH = INDEX_DIR / 'simplewiki_160w_dense_meta.joblib'

KELM_DENSE_INDEX_PATH = INDEX_DIR / 'kelm_500k_dense_hnsw.index'
KELM_DENSE_META_PATH = INDEX_DIR / 'kelm_500k_dense_meta.joblib'

for path in [PROJECT_ROOT, SIMPLEWIKI_CORPUS_PATH, KELM_CORPUS_PATH, INDEX_DIR]:
    print(path, 'exists=', path.exists())

usage = shutil.disk_usage(PROJECT_ROOT)
print(f'Drive visible space: free={usage.free / 1024**3:.2f} GB, total={usage.total / 1024**3:.2f} GB')

In [ ]:
import json
import time
from typing import Dict, Iterable, Tuple

import hnswlib
import joblib
import numpy as np
import torch
from sentence_transformers import SentenceTransformer

MODEL_NAME = 'sentence-transformers/multi-qa-MiniLM-L6-cos-v1'
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', DEVICE)
if DEVICE == 'cuda':
    print('GPU:', torch.cuda.get_device_name(0))

model = SentenceTransformer(MODEL_NAME, device=DEVICE)
DIM = model.get_sentence_embedding_dimension()
print('Embedding dim:', DIM)

## Funzioni di costruzione

- SimpleWiki: embedding text = `title + '. ' + chunk_text`.
- KELM: embedding text = `text`, oppure `sentence + '. ' + triple` se quei campi sono presenti.
- I metadata salvano sempre il testo originale, non il testo arricchito per embedding.

In [ ]:
def count_valid_rows(corpus_path: Path, limit: int | None = None) -> int:
    count = 0
    with corpus_path.open('r', encoding='utf-8') as inp:
        for line in inp:
            if not line.strip():
                continue
            row = json.loads(line)
            if str(row.get('text') or '').strip():
                count += 1
                if limit is not None and count >= limit:
                    break
    return count


def doc_metadata(row: Dict, default_source: str) -> Dict:
    return {
        'id': row.get('id'),
        'doc_id': row.get('doc_id') or row.get('id'),
        'chunk_id': row.get('chunk_id') if row.get('chunk_id') is not None else 0,
        'title': row.get('title', ''),
        'url': row.get('url', ''),
        'source': row.get('source', default_source),
        'text': str(row.get('text') or ''),
    }


def embedding_text(row: Dict, corpus_kind: str) -> str:
    text = str(row.get('text') or '').strip()
    title = str(row.get('title') or '').strip()
    if corpus_kind == 'simplewiki':
        return f'{title}. {text}' if title else text

    sentence = str(row.get('sentence') or '').strip()
    triple = str(row.get('triple') or '').strip()
    if sentence and triple:
        return f'{sentence}. {triple}'
    return text


def iter_batches(corpus_path: Path, corpus_kind: str, batch_size: int, limit: int | None = None):
    texts = []
    docs = []
    seen = 0
    with corpus_path.open('r', encoding='utf-8') as inp:
        for line in inp:
            if not line.strip():
                continue
            row = json.loads(line)
            text = str(row.get('text') or '').strip()
            if not text:
                continue

            texts.append(embedding_text(row, corpus_kind))
            docs.append(doc_metadata(row, corpus_kind))
            seen += 1

            if len(texts) >= batch_size:
                yield texts, docs
                texts, docs = [], []

            if limit is not None and seen >= limit:
                break

    if texts:
        yield texts, docs


def build_dense_hnsw(
    corpus_path: Path,
    corpus_kind: str,
    out_index_path: Path,
    out_meta_path: Path,
    limit: int | None = None,
    add_batch_size: int = 4096,
    encode_batch_size: int = 256,
    m: int = 32,
    ef_construction: int = 200,
    ef_search: int = 128,
):
    if not corpus_path.exists():
        raise FileNotFoundError(corpus_path)

    total = count_valid_rows(corpus_path, limit=limit)
    print(f'{corpus_kind}: valid rows = {total}')
    if total == 0:
        raise ValueError(f'No valid rows in {corpus_path}')

    index = hnswlib.Index(space='cosine', dim=DIM)
    index.init_index(max_elements=total, ef_construction=ef_construction, M=m)

    all_docs = []
    offset = 0
    started = time.time()

    for batch_texts, batch_docs in iter_batches(corpus_path, corpus_kind, add_batch_size, limit=limit):
        embeddings = model.encode(
            batch_texts,
            batch_size=encode_batch_size,
            convert_to_numpy=True,
            normalize_embeddings=True,
            show_progress_bar=False,
        ).astype('float32')

        ids = np.arange(offset, offset + len(batch_texts))
        index.add_items(embeddings, ids)
        all_docs.extend(batch_docs)
        offset += len(batch_texts)

        if offset == len(batch_texts) or offset % (add_batch_size * 10) == 0 or offset >= total:
            elapsed = time.time() - started
            rate = offset / elapsed if elapsed > 0 else 0
            print(f'{corpus_kind}: {offset}/{total} docs, {rate:.1f} docs/s')

    index.set_ef(ef_search)
    index.save_index(str(out_index_path))

    joblib.dump(
        {
            'kind': 'dense_hnsw',
            'model_name': MODEL_NAME,
            'hnsw_path': str(out_index_path),
            'docs': all_docs,
            'dim': DIM,
            'space': 'cosine',
            'ef': ef_search,
            'corpus_path': str(corpus_path),
            'corpus_kind': corpus_kind,
        },
        out_meta_path,
        compress=3,
    )

    print('Saved index:', out_index_path)
    print('Saved meta:', out_meta_path)
    return out_index_path, out_meta_path

## Smoke test su subset piccolo

Esegui prima questa cella: crea un indice KELM da 2k righe per verificare GPU, path e salvataggio. Poi puoi cancellare i due file `_smoke` da Drive.

In [ ]:
SMOKE = False

if SMOKE:
    build_dense_hnsw(
        corpus_path=KELM_CORPUS_PATH,
        corpus_kind='kelm',
        out_index_path=INDEX_DIR / 'kelm_2k_dense_hnsw_smoke.index',
        out_meta_path=INDEX_DIR / 'kelm_2k_dense_meta_smoke.joblib',
        limit=2000,
        add_batch_size=512,
        encode_batch_size=128,
    )

## 1. KELM dense index

Parti da KELM: e piu piccolo di SimpleWiki ed e il test migliore per capire tempi e spazio occupato.

In [ ]:
build_dense_hnsw(
    corpus_path=KELM_CORPUS_PATH,
    corpus_kind='kelm',
    out_index_path=KELM_DENSE_INDEX_PATH,
    out_meta_path=KELM_DENSE_META_PATH,
    add_batch_size=4096,
    encode_batch_size=256,
    m=32,
    ef_construction=200,
    ef_search=128,
)

## 2. SimpleWiki dense index

Esegui questa cella solo se KELM e andato a buon fine e hai spazio sufficiente su Drive. SimpleWiki richiede molto piu tempo, RAM e spazio.

In [ ]:
RUN_SIMPLEWIKI = False

if RUN_SIMPLEWIKI:
    build_dense_hnsw(
        corpus_path=SIMPLEWIKI_CORPUS_PATH,
        corpus_kind='simplewiki',
        out_index_path=SIMPLEWIKI_DENSE_INDEX_PATH,
        out_meta_path=SIMPLEWIKI_DENSE_META_PATH,
        add_batch_size=4096,
        encode_batch_size=256,
        m=32,
        ef_construction=200,
        ef_search=128,
    )

## Query test

In [ ]:
def load_dense_index(meta_path: Path):
    meta = joblib.load(meta_path)
    idx = hnswlib.Index(space=meta['space'], dim=meta['dim'])
    idx.load_index(meta['hnsw_path'])
    idx.set_ef(meta.get('ef', 128))
    return meta, idx


def dense_search(query: str, meta_path: Path, top_k: int = 5):
    meta, idx = load_dense_index(meta_path)
    query_embedding = model.encode(
        [query],
        convert_to_numpy=True,
        normalize_embeddings=True,
    ).astype('float32')
    labels, distances = idx.knn_query(query_embedding, k=top_k)
    for label, distance in zip(labels[0], distances[0]):
        doc = meta['docs'][int(label)]
        score = 1.0 - float(distance)
        text = doc.get('text', '')
        if len(text) > 300:
            text = text[:297] + '...'
        print(f'score={score:.4f} source={doc.get("source", "")} title={doc.get("title", "")}')
        print(text)
        print()


dense_search('Which city is the capital of France?', KELM_DENSE_META_PATH, top_k=5)